In [1]:
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil
%config InlineBackend.figure_format = 'retina'
sns.set_theme()

# 1. Data exploration and initial analysis

In [2]:
df_raw = pd.read_csv("../data/acdc_dataset.csv", sep=";", index_col="member_id")
df = df_raw.copy()
df_raw.head()


,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,membership_end,...,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value
member_id,,,,,,,,,,,,,,,,,,,,,
CW000001,38.0,Female,Information Technology,Portugal,False,0,flex,3,2021-08-01,2023-03-06,...,False,True,0,False,True,True,False,False,2,4477.97
CW000002,38.0,Male,Information Technology,Portugal,True,2,HOT_DESK,2,2023-11-24,1970-01-01,...,False,False,0,True,False,False,False,False,0,165.86
CW000003,43.0,Male,Education,DE,True,1,flex,3,2021-12-03,1970-01-01,...,False,False,0,False,False,False,False,False,1,4486.17
CW000004,38.0,F,Legal,portugal,False,0,flex,3,2022-01-02,2022-02-15,...,False,False,0,False,False,False,False,False,0,258.63
CW000005,45.0,Female,Information Technology,United Kingdom,False,0,hot_desk,2,2023-10-24,1970-01-01,...,False,False,0,False,False,True,False,False,0,335.74


In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15465 entries, CW000001 to CW015101
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          14907 non-null  float64
 1   gender                       14935 non-null  object 
 2   industry                     15465 non-null  object 
 3   country                      15035 non-null  object 
 4   is_team_account              15465 non-null  bool   
 5   n_colleagues_on_account      15465 non-null  int64  
 6   plan_tier                    15360 non-null  object 
 7   plan_days_per_week           15465 non-null  int64  
 8   membership_start             15465 non-null  object 
 9   membership_end               15465 non-null  object 
 10  last_badge_swipe             15465 non-null  object 
 11  days_since_last_visit        15465 non-null  int64  
 12  total_visits                 15465 non-null  int64  
 13  avg_weekly_

In [4]:
df_raw.columns

Index(['age', 'gender', 'industry', 'country', 'is_team_account',
       'n_colleagues_on_account', 'plan_tier', 'plan_days_per_week',
       'membership_start', 'membership_end', 'last_badge_swipe',
       'days_since_last_visit', 'total_visits', 'avg_weekly_visits',
       'entitled_visits_last_period', 'actual_visits_last_period',
       'meeting_room_hours', 'event_space_hours', 'guest_passes_issued',
       'uses_meeting_rooms', 'uses_event_space', 'uses_guest_passes',
       'uses_print', 'uses_mail_handling', 'uses_phone_booths',
       'uses_day_passes', 'uses_storage', 'uses_bike_locker', 'n_renewals',
       'lifetime_value'],
      dtype='object')

In [5]:
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
age,14907.0,34.858724,10.394649,1.0,29.00,34.00,40.00,149.00
n_colleagues_on_account,15465.0,0.537019,1.177668,-1.0,0.00,0.00,1.00,9.00
plan_days_per_week,15465.0,3.387262,1.458818,0.0,2.00,3.00,5.00,9.00
days_since_last_visit,15465.0,117.109279,275.532807,-956.0,4.00,13.00,55.00,1212.00
total_visits,15465.0,110.406531,182.939340,-112.0,8.00,34.00,132.00,6530.00
avg_weekly_visits,15465.0,1.777214,1.937515,0.0,0.71,1.42,2.41,47.40
entitled_visits_last_period,15465.0,27.517393,13.480235,0.0,17.39,26.09,43.48,78.27
actual_visits_last_period,15465.0,13.800970,11.387207,0.0,5.00,10.00,20.00,60.00
meeting_room_hours,14167.0,33.075570,428.169678,-56.6,0.00,0.00,7.20,41220.00
event_space_hours,15465.0,5.117220,26.013097,-13.5,0.00,0.00,0.00,602.80


In [6]:
df_raw.describe(include="bool").T

,count,unique,top,freq
is_team_account,15465,2,False,11112
uses_meeting_rooms,15465,2,False,10202
uses_event_space,15465,2,False,13624
uses_guest_passes,15465,2,False,12651
uses_mail_handling,15465,2,False,11901
uses_phone_booths,15465,2,False,11469
uses_day_passes,15465,2,False,12069
uses_storage,15465,2,False,13169
uses_bike_locker,15465,2,False,15439


In [7]:
df_raw.describe(include="object").T
#plan tier should only have 5 unique values
#gender shou have 3: Female, Male, Other

,count,unique,top,freq
gender,14935,12,Female,7414
industry,15465,43,Information Technology,3322
country,15035,33,Portugal,9959
plan_tier,15360,19,flex,5182
membership_start,15465,1539,2021-01-01,560
membership_end,15465,1423,1970-01-01,3956
last_badge_swipe,15465,1543,2023-12-30,279
lifetime_value,14425,11860,0,1725


# 2. Data quality checks

In [8]:
df['membership_start'] = pd.to_datetime(df["membership_start"], format='mixed')
df['membership_end'] = pd.to_datetime(df["membership_end"], format='mixed')
df['last_badge_swipe'] = pd.to_datetime(df["last_badge_swipe"], format='mixed')

We turned the variable `lifetime_value` to a float. The variable had symbols that made them into an object type.

In [9]:
df["lifetime_value"] = (
    df["lifetime_value"]
    .str.replace("€", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(float)
)

In [10]:
df_raw["uses_print"].unique()

array([0, 1])

`uses_print` only has the values 1 and 0, so we turn it into a boolean type.

In [11]:
df["uses_print"] = df["uses_print"].astype("boolean")
df["uses_print"].dtype

BooleanDtype

In [12]:
df.isna().sum()[lambda s: s > 0]

age                     558
gender                  530
country                 430
plan_tier               105
meeting_room_hours     1298
guest_passes_issued    1271
lifetime_value         1040
dtype: int64

In [13]:
#Turning object values to lower case
cols_object = df.select_dtypes(include="object").columns
df[cols_object] = df[cols_object].apply(lambda s: s.str.lower())

print(f"Gender unique values: {df["gender"].unique()}")
print(f"Country unique values: {df["country"].unique()}")
print(f"Plan tier unique values: {df["plan_tier"].unique()}")
print(f"Industry unique values: {df["industry"].unique()}")

Gender unique values: ['female' 'male' 'f' 'other' nan ' female' 'm' ' male' ' other' 'o']
Country unique values: ['portugal' 'de' 'united kingdom' 'spain' 'germany' 'netherlands' 'france'
 'pt' 'other (europe)' 'portucal' nan 'uk' 'ireland' 'ie' 'portugal '
 'deutschland' 'u.k.' 'gb' 'es' 'nl' 'españa' 'espana' 'the netherlands'
 'fr' 'holland']
Plan tier unique values: ['flex' 'hot_desk' 'dedicated_desk' 'full_time' 'private_office'
 'hot desk' 'dedicateddesk' nan 'hotdesk' 'fulltime' 'privateoffice'
 'full time' 'dedicated desk' 'private office']
Industry unique values: ['information technology' 'education' 'legal' 'health' 'unknown' 'other'
 'consulting' 'design & creative' 'finance' 'marketing & pr'
 'marketing and pr' 'real estate' 'marketing/pr' 'design/creative'
 'realestate' 'it' 'tech' 'design and creative' 'real-estate']


In [ ]:
#Replacing gender values
df_raw["gender"] = df_raw["gender"].replace([" female", "f"], "female")
df_raw["gender"] = df_raw["gender"].replace([" male", "m"], "male")
df_raw["gender"] = df_raw["gender"].replace([" other", "o"], "other")

#Replacing country values
df_raw["country"] = df_raw["country"].replace({
    "portucal":"portugal", "portugal ":"portugal", "pt":"portugal",
    "de":"germany", "deutschland":"germany",
    "uk":"united kingdom", "u.k.":"united kingdom", "gb":"united kingdom",
    "es":"spain", "españa":"spain", "espana":"spain",
    "nl":"netherlands", "the netherlands":"netherlands", "holland":"netherlands",
    "fr":"france",
    "ie":"ireland"})

#Replacing Plan tier values
df_raw["plan_tier"] = df_raw["plan_tier"].replace(["hotdesk", "hot desk"], "hot_desk")
df_raw["plan_tier"] = df_raw["plan_tier"].replace(["dedicateddesk", "dedicated desk"], "dedicated_desk")
df_raw["plan_tier"] = df_raw["plan_tier"].replace(["fulltime", "full time"], "full_time")
df_raw["plan_tier"] = df_raw["plan_tier"].replace(["privateoffice", "private office"], "private_office")

#Replacing Industry values
df_raw["industry"] = df_raw["industry"].replace(["it", "tech"], "information technology")
df_raw["industry"] = df_raw["industry"].replace(["design & creative", "design/creative"], "design and creative")
df_raw["industry"] = df_raw["industry"].replace(["marketing & pr", "marketing/pr"], "marketing and pr")
df_raw["industry"] = df_raw["industry"].replace(["realestate", "real-estate"], "real estate")

df_raw.describe(include="object").T

,count,unique,top,freq
gender,14935,3,female,7867
industry,15465,11,information technology,3559
country,15035,13,portugal,10378
plan_tier,15360,5,flex,5453


In [15]:
df.duplicated().sum()

np.int64(293)

In [16]:
df_nodups = df.drop_duplicates()

In [17]:
non_metric_features = ['gender', 'country', 'industry', 'plan_tier', 'is_team_account',
 'uses_meeting_rooms', 'uses_event_space', 'uses_guest_passes',
 'uses_print', 'uses_mail_handling', 'uses_phone_booths', 'uses_day_passes',
 'uses_storage', 'uses_bike_locker']

metric_features = df_nodups.columns.drop(non_metric_features).to_list()